# RDD Dependencies, Lineage Graphs, and Fault Tolerance

**Module**: 02_rdd  
**Notebook**: 04_rdd_dependencies_and_lineage.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Spark constructs RDD lineage graphs, the distinction between Narrow and Wide dependencies, and how lineage guarantees fault tolerance.
- **Why does this topic exist?**: Lineage is the foundation of Spark's resilience. Understanding dependencies explains why shuffles exist and how lost data is reconstructed.
- **What problem does it solve?**: Without understanding lineage, developers cannot predict stage boundaries or diagnose recomputation overhead.
- **Where is it used in real-world Data Engineering?**: Explaining lineage recomputation and dependency classifications is a mandatory component of Spark technical interviews.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | RDD partitions |
| **Required depth** | Working knowledge: can read lineage and explain recomputation |
| **Data Engineer** | Read `toDebugString()`, identify narrow/wide dependencies, and explain lineage-based recovery. |
| **Senior Data Engineer** | Decide when to checkpoint and how to keep lineage deterministic. |
| **Spark specialist** | Reason about shuffle dependencies, stage boundaries, and lineage truncation. |
| **Next topic** | `02_rdd/05_rdd_caching_and_persistence.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Lineage is Spark's fault-tolerance mechanism: if a partition is lost, Spark recomputes it from its parents.
That is powerful but not free. Long chains of transformations can make recomputation expensive, and
non-deterministic code can make the recomputed result differ from the original. Checkpointing and deterministic
transformations are the practical safeguards.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Lineage** | The recipe for rebuilding an RDD | Directed graph of parent dependencies |
| **Narrow dependency** | Parent partition feeds one child partition | Pipelined within a stage |
| **Wide dependency** | Parent partitions feed multiple child partitions | Shuffle and stage boundary |
| **Stage boundary** | Where a shuffle occurs | The scheduler materializes map outputs |
| **Recomputation** | Rebuild lost data from parents | Triggered by executor or block loss |
| **Checkpoint** | Truncate lineage with durable storage | Writes data and cuts the dependency chain |

### How it works, step by step

1. The driver tracks RDD dependencies as a DAG.
2. A narrow dependency keeps the pipeline in one stage.
3. A wide dependency creates a shuffle and a new stage.
4. If a partition is lost, the scheduler walks the lineage to the nearest available parent.
5. Shuffle map outputs may need recomputation if the map stage is lost.
6. Checkpointing writes the dataset to reliable storage and replaces the lineage with a checkpoint reference.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Narrow vs Wide Dependencies in the Lineage Graph

```
NARROW DEPENDENCIES (Pipelined in 1 Stage, No Network Transfer):
Parent Partitions:   [P0]    [P1]    [P2]
                       |         |         |  (e.g. map, filter)
                       v         v         v
Child Partitions:    [C0]    [C1]    [C2]

WIDE DEPENDENCIES (Shuffle, Stage Boundary, Network Transfer):
Parent Partitions:   [P0]    [P1]    [P2]
                        \   \  /   \  /   /   (e.g. groupByKey, join)
                         v   v v   v v   v
Child Partitions:       [C0]     [C1]
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("RDD_Lineage_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Inspecting RDD Lineage with `toDebugString()`

In [ ]:
sc = spark.sparkContext

# Build a multi-step RDD pipeline
base_rdd = sc.textFile("/opt/spark-data/retail/orders.csv")
header = base_rdd.first()

cleaned = base_rdd.filter(lambda line: line != header)
mapped = cleaned.map(lambda line: line.split(","))
keyed = mapped.map(lambda cols: (cols[7], 1)) # key by order status
reduced = keyed.reduceByKey(lambda a, b: a + b) # WIDE transformation

print("=== RDD Lineage Graph (toDebugString) ===")
print(reduced.toDebugString().decode('utf-8'))

## 4. Analyzing Stage Boundaries in Lineage Output

Notice the `ShuffledRDD` entry above: that is where Spark places a **Stage Boundary**!

In [ ]:
# Inspect dependencies directly using Python introspection
print(f"reduced RDD dependencies count: {len(reduced.dependencies)}")
for dep in reduced.dependencies:
    print(f"  Dependency Type: {type(dep).__name__}")
    print(f"  Parent RDD:      {dep.rdd}")

---

## Experiment: Verifying Partition Recomputation from Lineage

- **1. What are we testing?**: Demonstrate that evaluating an action on an unpersisted RDD re-executes its entire upstream lineage from disk.
- **2. What do we expect?**: Each action will re-execute the map operations independently.

In [ ]:
call_counter = sc.accumulator(0)

def tracked_map(x):
    global call_counter
    call_counter.add(1)
    return x * 2

rdd_calc = sc.parallelize(range(10), 2).map(tracked_map)

# First Action
print("Action 1 count:", rdd_calc.count())
print(f"Accumulator after Action 1: {call_counter.value} (Expected 10)")

# Second Action on same unpersisted RDD
print("Action 2 sum:  ", rdd_calc.sum())
print(f"Accumulator after Action 2: {call_counter.value} (Expected 20 due to recomputation!)")

**3. What happened?**: The accumulator reached 20 instead of staying at 10.  
**4. Why did it happen?**: Because rdd_calc was not cached, Spark recomputed its entire lineage graph from scratch for Action 2.  
**5. Practical Engineering Lesson**: Uncached RDDs/DataFrames re-evaluate their complete transformation graph on every single action.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Audit trail for regulatory reporting

- **Industry**: Banking
- **Scenario**: A bank must reproduce a daily risk report months later from immutable source data.
- **Problem**: The report pipeline has a long chain of transformations, and a recomputation must produce the same result.
- **Why this concept is relevant**: Lineage shows exactly which inputs and transformations produced each partition; deterministic code makes replay reliable.
- **Implementation idea**: Persist raw inputs, use deterministic transformations, checkpoint expensive intermediate results, and version the job code.
- **Expected result**: A reproducible report with a documented lineage and checkpoint strategy.
- **Alternative**: A warehouse with time travel and SQL versioning.
- **Trade-offs**: Spark lineage is execution-level; warehouse time travel is storage-level and often easier for auditors.

### Use Case 2: Recovering from a lost executor

- **Industry**: E-commerce
- **Scenario**: A worker dies during a long aggregation and the job continues without restarting.
- **Problem**: Lost partitions must be rebuilt without corrupting the result.
- **Why this concept is relevant**: Spark recomputes only the lost partitions from lineage, then retries the affected tasks.
- **Implementation idea**: Monitor task retries and stage recomputation in the Spark UI; keep transformations deterministic and avoid side effects.
- **Expected result**: The job completes with the same output, at the cost of recomputation time.
- **Alternative**: A replicated storage layer with no recomputation.
- **Trade-offs**: Lineage is cheap when the pipeline is narrow; replication is simpler for expensive shuffle stages.

---

## Business Value

- Improves reliability without full data replication.
- Reduces recovery time for transient failures.
- Makes data provenance and audit discussions concrete.
- Can increase runtime when recomputation is expensive.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Keep transformations deterministic; avoid random values without a fixed seed and side effects in `map`.
- Checkpoint long lineages and iterative jobs.
- Monitor task retries and failed stages, not just final success.
- Remember that checkpointing writes data; choose durable storage and manage retention.
- Do not rely on lineage for data that must survive the application; write it out.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Read a lineage graph

**Steps**

1. Build a chain: `parallelize → filter → map → reduceByKey`.
2. Print `toDebugString()`.
3. Identify the narrow and wide dependencies.
4. Compare the stage count with the Spark UI.

**What to observe**: `reduceByKey` introduces a shuffle boundary; the graph shows the parent partitions.

**Healthy result**: You can point to the exact transformation that caused the stage boundary.

**Problematic result**: A long chain of narrow transformations is pipelined, but a hidden shuffle breaks the pipeline.

**Engineering takeaway**: Lineage is both a recovery plan and a performance map.

---

## Trade-offs and Alternatives

### Advantages

- No eager replication
- Fine-grained recovery
- Transparent provenance

### Limitations

- Recomputation can be expensive
- Non-deterministic code breaks reproducibility
- Long lineages can overflow or slow planning

### When to use it

- Fault tolerance
- Audit and replay
- Iterative algorithms with checkpoints

### When not to use it

- Relying on lineage for permanent storage
- Side-effecting transformations
- Unbounded lineage growth

### Alternatives inside Spark

- Checkpointing
- Persist/cache
- AQE

### Alternatives outside Spark

- Replicated storage
- Database transactions
- Time-travel tables

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify OneToOneDependency
**Problem Statement**: Create an RDD with map and filter, and inspect its dependencies to identify OneToOneDependency.  
**Input / Context**: sc.parallelize(range(10)).map(lambda x: x+1).filter(lambda x: x > 5)

In [ ]:
# Exercise 1: OneToOneDependency

### Exercise 2 (Intermediate): Trace Stage Count in toDebugString
**Problem Statement**: Write a pipeline that performs two distinct shuffles (e.g. reduceByKey followed by sortByKey) and count ShuffledRDD occurrences in toDebugString.  
**Input / Context**: reduced RDD

In [ ]:
# Exercise 2: Count shuffles in lineage

### Exercise 3 (Advanced): Simulate Lineage Recomputation
**Problem Statement**: Build an RDD pipeline from retail/products.csv, take an action, and print lineage depth.  
**Input / Context**: retail/products.csv

In [ ]:
# Exercise 3: Lineage depth

---

## Production Debugging Exercise

### Non-deterministic recomputation

**Scenario**: A job uses `random()` inside a `map` without a seed; after an executor failure, the recomputed partitions differ.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Look for random or time-dependent functions in transformations.
2. Use a seeded random function or a deterministic hash.
3. Check whether the output is used for audit or reconciliation.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify OneToOneDependency):
```python
r = sc.parallelize(range(10)).map(lambda x: x+1).filter(lambda x: x > 5)
print("Dependencies of filter step:", [type(d).__name__ for d in r.dependencies])
```

#### Solution 2 (Trace Stage Count in toDebugString):
```python
sorted_rdd = reduced.sortByKey()
debug_str = sorted_rdd.toDebugString().decode('utf-8')
print("Lines with ShuffledRDD:")
for line in debug_str.split("\n"):
    if"ShuffledRDD"in line:
        print("->", line)
```

#### Solution 3 (Simulate Lineage Recomputation):
```python
prods = sc.textFile("/opt/spark-data/retail/products.csv").filter(lambda l:"Electronics"in l).map(lambda l: l.split(",")).map(lambda c: (c[2], float(c[3]))).reduceByKey(lambda a, b: a + b)
print(f"Lineage tree:\n{prods.toDebugString().decode('utf-8')}")
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Lineage Graph**: Spark tracks the sequence of transformations so lost partitions can be recomputed deterministically.
- **Narrow Dependency**: 1:1 partition mapping; pipelined within a single stage.
- **Wide Dependency**: Shuffle required; draws stage boundaries in the DAG.

### Essential Technical Terms
- `Lineage`: Deterministic acyclic graph of parent RDD transformations.
- `OneToOneDependency`: Narrow dependency where child partition depends only on 1 parent partition.
- `ShuffleDependency`: Wide dependency where child partition depends on all parent partitions.

### Common Engineering Mistakes
- **Assuming Spark Caches Automatically**: Spark does NOT cache intermediate RDDs; every action recomputes from source unless persisted.
- **Ignoring Lineage Length**: Extremely long iterative lineage graphs can cause JVM StackOverflowError (fix with checkpointing).

### Interview Questions on This Topic
- **Q: How does Spark achieve fault tolerance?**
  - *A*: Through RDD lineage recomputation. If an executor fails, Spark recomputes only the missing partition from its parent dependencies.
- **Q: What is toDebugString()?**
  - *A*: An RDD method that displays the full dependency lineage tree.




### More Interview Questions

- **Q: What is the difference between a narrow and a wide dependency?**
  - *A*: A narrow dependency uses at most one child partition per parent partition and can be pipelined; a wide dependency requires a shuffle because multiple child partitions depend on the same parent partition.
- **Q: When should you checkpoint instead of relying on lineage?**
  - *A*: When the lineage is long or expensive to recompute, when the job is iterative, or when the result must survive application restarts.

### What to Study Next
Open 02_rdd/05_rdd_caching_and_persistence.ipynb to learn caching and memory storage levels.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')